# Spike: can `ncu` (Nsight Compute) profile a kernel on a Colab T4?

**What this proves:** the entire GPU-kernel-advisor tool depends on reading real hardware
performance counters via `ncu` inside Colab. This notebook compiles a trivial vector-add
kernel, profiles it, and parses the metrics to JSON — the exact path the backend will use.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

**Success looks like:** Cell 5 prints `VERDICT: OK`, Cell 6 shows a metric table with real
numbers, Cell 7 prints a JSON dict of metrics.

**Failure looks like:** `==ERROR== ERR_NVGPUCTRPERM` — the driver is restricting GPU
performance counters and we fall back (see final cell).


## 1. Environment check

Three things matter:
1. `nvcc` present (it is, preinstalled under `/usr/local/cuda`).
2. We run as **root** — this is why profiling is likely to work.
3. The driver parameter `RmProfilingAdminOnly`: if `1`, perf counters are restricted to
   root/`CAP_SYS_ADMIN`. Since Colab cells run as root, even `1` is usually fine. The only
   truly bad sign is an `ERR_NVGPUCTRPERM` at actual profile time (Cell 5 tests this
   directly — that's the ground truth, not this cell).


In [ ]:
%%bash
nvidia-smi
echo "-----"
nvcc --version | tail -1
echo "-----"
echo "running as: $(whoami)"
echo "-----"
grep -i RmProfilingAdminOnly /proc/driver/nvidia/params || echo "RmProfilingAdminOnly not exposed in /proc (not necessarily a problem)"


## 2. Locate or install `ncu`

Colab images usually ship Nsight Compute with the CUDA toolkit. If it's missing, Colab
already has NVIDIA's apt repo configured, so `cuda-nsight-compute-<ver>` installs cleanly.


In [ ]:
%%bash
if command -v ncu >/dev/null; then
    echo "ncu already on PATH: $(command -v ncu)"
else
    found=$(ls /usr/local/cuda*/bin/ncu /opt/nvidia/nsight-compute/*/ncu 2>/dev/null | head -1 || true)
    if [ -n "$found" ]; then
        ln -sf "$found" /usr/local/bin/ncu
        echo "found and linked: $found"
    else
        # install the package matching the toolkit version, e.g. cuda-nsight-compute-12-4
        CUDA_VER=$(nvcc --version | grep -oP 'release \K[0-9]+\.[0-9]+' | tr . -)
        echo "ncu not found; installing cuda-nsight-compute-${CUDA_VER} ..."
        apt-get -qq update
        apt-get -qq install -y cuda-nsight-compute-${CUDA_VER}
        ln -sf /usr/local/cuda/bin/ncu /usr/local/bin/ncu 2>/dev/null || true
    fi
fi
echo "-----"
ncu --version


## 3. Minimal kernel: vector add

Deliberately boring and analytically predictable — we know exactly what every metric
*should* read, so wrong-looking numbers mean a broken profiling path, not a mystery kernel.


In [ ]:
%%writefile vector_add.cu
#include <cstdio>
#include <cuda_runtime.h>

#define CUDA_CHECK(call)                                                      \
    do {                                                                      \
        cudaError_t err_ = (call);                                            \
        if (err_ != cudaSuccess) {                                            \
            fprintf(stderr, "CUDA error: %s at %s:%d\n",                      \
                    cudaGetErrorString(err_), __FILE__, __LINE__);            \
            return 1;                                                         \
        }                                                                     \
    } while (0)

__global__ void vecAdd(const float* __restrict__ a,
                       const float* __restrict__ b,
                       float* __restrict__ c, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main()
{
    // 16M elements: 64 MB per array, ~192 MB of DRAM traffic per launch.
    // Big enough that DRAM throughput numbers are stable and meaningful.
    const int n = 1 << 24;
    const size_t bytes = (size_t)n * sizeof(float);

    float *a, *b, *c;
    CUDA_CHECK(cudaMalloc(&a, bytes));
    CUDA_CHECK(cudaMalloc(&b, bytes));
    CUDA_CHECK(cudaMalloc(&c, bytes));

    float* h = (float*)malloc(bytes);
    for (int i = 0; i < n; ++i) h[i] = 1.0f;
    CUDA_CHECK(cudaMemcpy(a, h, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(b, h, bytes, cudaMemcpyHostToDevice));

    const int block = 256;
    const int grid  = (n + block - 1) / block;
    vecAdd<<<grid, block>>>(a, b, c, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(h, c, bytes, cudaMemcpyDeviceToHost));
    for (int i = 0; i < n; ++i) {
        if (h[i] != 2.0f) { fprintf(stderr, "FAIL at %d: %f\n", i, h[i]); return 1; }
    }
    printf("PASS (n=%d)\n", n);
    return 0;
}


In [ ]:
%%bash
nvcc -O3 -arch=sm_75 -lineinfo vector_add.cu -o vector_add
./vector_add


## 4. The permission probe

The smallest possible `ncu` invocation: one cheap metric, one kernel launch. This is the
go/no-go moment — it either returns a number or `ERR_NVGPUCTRPERM`.


In [ ]:
%%bash
set +e
ncu --metrics sm__cycles_elapsed.avg --launch-count 1 ./vector_add > /tmp/probe.log 2>&1
status=$?
cat /tmp/probe.log
echo "-----"
if grep -q ERR_NVGPUCTRPERM /tmp/probe.log; then
    echo "VERDICT: BLOCKED — perf counters restricted on this runtime. See fallback ladder in the last cell."
elif [ $status -eq 0 ] && grep -q sm__cycles_elapsed /tmp/probe.log; then
    echo "VERDICT: OK — ncu can read performance counters on this runtime."
else
    echo "VERDICT: UNCLEAR — ncu failed for a different reason (exit $status). Read the log above; likely a version/driver mismatch, not permissions."
fi


## 5. Full metric run (human-readable)

The metrics the advisor actually needs (modern names; `nvprof`-era names are dead):

| what you want | ncu metric |
|---|---|
| kernel duration | `gpu__time_duration.sum` |
| achieved occupancy | `sm__warps_active.avg.pct_of_peak_sustained_active` |
| compute throughput, % of peak (SOL SM) | `sm__throughput.avg.pct_of_peak_sustained_elapsed` |
| DRAM throughput, % of peak (SOL DRAM) | `dram__throughput.avg.pct_of_peak_sustained_elapsed` |
| DRAM traffic (bytes) — roofline x-axis input | `dram__bytes.sum` |
| absolute DRAM bandwidth | `dram__bytes.sum.per_second` |
| warp execution efficiency | `smsp__thread_inst_executed_per_inst_executed.ratio` (÷32 → efficiency) |
| FP32 add FLOPs — roofline y-axis input | `smsp__sass_thread_inst_executed_op_fadd_pred_on.sum` |


In [ ]:
%%bash
ncu --kernel-name vecAdd --launch-count 1 \
    --metrics gpu__time_duration.sum,\
sm__warps_active.avg.pct_of_peak_sustained_active,\
sm__throughput.avg.pct_of_peak_sustained_elapsed,\
dram__throughput.avg.pct_of_peak_sustained_elapsed,\
dram__bytes.sum,\
dram__bytes.sum.per_second,\
smsp__thread_inst_executed_per_inst_executed.ratio,\
smsp__sass_thread_inst_executed_op_fadd_pred_on.sum \
    ./vector_add


## 6. CSV → JSON parse (proves the backend's parsing path)

Same run with `--csv`, parsed into the dict shape the FastAPI endpoint will return.


In [ ]:
import io, json, subprocess
import pandas as pd

METRICS = [
    "gpu__time_duration.sum",
    "sm__warps_active.avg.pct_of_peak_sustained_active",
    "sm__throughput.avg.pct_of_peak_sustained_elapsed",
    "dram__throughput.avg.pct_of_peak_sustained_elapsed",
    "dram__bytes.sum",
    "dram__bytes.sum.per_second",
    "smsp__thread_inst_executed_per_inst_executed.ratio",
    "smsp__sass_thread_inst_executed_op_fadd_pred_on.sum",
]

res = subprocess.run(
    ["ncu", "--csv", "--kernel-name", "vecAdd", "--launch-count", "1",
     "--metrics", ",".join(METRICS), "./vector_add"],
    capture_output=True, text=True,
)

# The CSV block starts at the header row (first field is "ID"). Everything
# above it is ncu log lines plus the program's own stdout.
lines = res.stdout.splitlines()
header_idx = next(
    (i for i, l in enumerate(lines) if l.startswith('"ID"') or l.startswith("ID,")),
    None,
)
if header_idx is None:
    print(res.stdout)
    print(res.stderr)
    raise RuntimeError("No CSV header found — raw ncu output printed above")

df = pd.read_csv(io.StringIO("\n".join(lines[header_idx:])), thousands=",")

if "Metric Name" in df.columns:
    # long layout: one row per (kernel, metric)
    metrics = {
        row["Metric Name"]: {"value": row["Metric Value"],
                             "unit": row.get("Metric Unit", "")}
        for _, row in df.iterrows()
    }
else:
    # wide layout: one row per kernel launch, one column per metric
    row = df.iloc[0]
    metrics = {m: {"value": row[m]} for m in df.columns if m in METRICS}

if not metrics:
    raise RuntimeError(f"Parsed CSV but matched no metrics. Columns: {list(df.columns)}")

print(json.dumps(metrics, indent=2, default=str))


## 7. Interpreting the numbers (sanity check)

Vector add on a T4 (16M floats, 256 threads/block) should land roughly here:

- **Duration:** ~0.8–1.5 ms (192 MB of traffic at ~200–280 GB/s achieved).
- **Achieved occupancy:** high, ~70–90%. A trivially latency-bound kernel like this has no
  occupancy limiter (low registers, no shared memory).
- **DRAM SOL:** ~60–85% of peak — the kernel is purely memory-bound, so this should be the
  dominant number.
- **SM SOL:** low, well under 25%. If SM% ≈ DRAM% something is off.
- **Threads per instruction:** ~32.0 (no divergence). Divide by 32 for the classic
  "warp execution efficiency" ≈ 100%.
- **`fadd` count:** exactly 16,777,216 (one add per element).
- **`dram__bytes.sum`:** ≈ 201 MB (2 arrays read + 1 written = 192 MiB, plus a little
  metadata traffic).

Roofline sanity: AI = 16.78 MFLOP / ~201 MB ≈ **0.083 FLOP/byte**, far left of the T4 ridge
point (8.1 TFLOP/s FP32 ÷ 320 GB/s ≈ 25 FLOP/byte) → firmly memory-bound. If your parsed
numbers reproduce this story, the whole metrics pipeline is trustworthy.

## If BLOCKED (`ERR_NVGPUCTRPERM`): fallback ladder

You **cannot** fix this inside Colab — the module param is set on the host and the nvidia
module can't be reloaded from the guest. In order of preference:

1. **Kaggle notebooks** (free T4/P100, also root, has nvcc): run this exact notebook there.
2. **Cheap rented instance with real root** — RunPod / Vast.ai / Lambda, T4 or A10 at
   ~$0.10–0.30/hr. Set `NVreg_RestrictProfilingToAdminUsers=0` (or just profile as root).
   This preserves the entire architecture; only the tunnel origin changes. **Recommended
   over contorting the product around missing counters.**
3. **nsys** works without counter permissions but only gives timeline data (kernel duration,
   memcpys) — no occupancy, no DRAM throughput, no divergence. You'd have to derive
   "achieved bandwidth" analytically (bytes you *think* the kernel moves ÷ measured time),
   but the access pattern is exactly the thing you're trying to diagnose. Big product
   downgrade; last resort.
4. **Timing + occupancy API only** (`cudaOccupancyMaxActiveBlocksPerMultiprocessor` +
   CUDA events): keeps a demo alive but the reports stop being "grounded in hardware
   counters", which is the tool's whole pitch.
